# Shared Variables in Spark

Normally, every Executor works independently.

Variables created in the Driver are **not automatically shared** with Executors.

Spark provides **Shared Variables** to safely share information between the Driver and Executors.

There are two types of Shared Variables:

1. Broadcast Variable
2. Accumulator

# Spark Shared Variables

Spark has two types of Shared Variables:

```
          Shared Variables
                 │
        ┌────────┴────────┐
        │                 │
        ▼                 ▼
 Broadcast Variable   Accumulator
(Read Only)          (Write Only)
```
# Broadcast vs Accumulator

```
               DRIVER
                  │
      ┌───────────┴────────────┐
      │                        │
      ▼                        ▲
Broadcast Variable        Accumulator

Driver ─────────────▶ Executors
(Read Only)

Executors ──────────▶ Driver
(Add Only)
```

---

# Why Do We Need Shared Variables?

Imagine you have:

```
Driver
   │
   ▼
Worker 1
Worker 2
Worker 3
Worker 4
```

Each worker has its own memory.

If you create:

```python
country = "India"
```

Every executor gets its own copy.

If the object is large (for example, a lookup dictionary with 1 million records), Spark would send that object repeatedly to every task, wasting network bandwidth and memory.

To solve this, Spark provides **Broadcast Variables**.

Similarly, if you want to count invalid records across all executors, Spark provides **Accumulators**.

---

# 1. Broadcast Variable

## Simple Definition

A Broadcast Variable is a **read-only variable** that the Driver sends **once** to every Executor.

Instead of sending the same data repeatedly for every task, Spark sends it only once and stores it in each Executor's memory.

---

## Real-Life Analogy

Imagine a classroom.

Teacher = Driver

Students = Executors

The teacher gives each student the same textbook.

Instead of giving a new textbook for every question, each student keeps one copy.

That's exactly how Broadcast Variables work.

---

## Architecture

```text
                 Driver
                    │
      Broadcast Dictionary
                    │
     ───────────────┼───────────────
        │           │            │
        ▼           ▼            ▼
   Executor1   Executor2   Executor3

Each Executor stores one copy.
```

---

# Example

Suppose you have

### Sales Table

| CustomerID | CountryCode |
|------------|-------------|
|101|US|
|102|IN|
|103|UK|

---

### Country Lookup

| Code | Country |
|------|----------|
|US|United States|
|IN|India|
|UK|United Kingdom|

Instead of joining a tiny lookup table repeatedly,

broadcast it.

```python
country_dict = {
    "US":"United States",
    "IN":"India",
    "UK":"United Kingdom"
}

broadcast_country = spark.sparkContext.broadcast(country_dict)
```

Now every Executor has one local copy.

---

# Accessing Broadcast Variable

```python
broadcast_country.value["IN"]
```

Output

```
India
```

---

# Real-Time Project Example

Suppose you're processing

```
500 Million Sales Records
```

You also have

```
Country Lookup

Only 200 rows
```

Without Broadcast

Every Executor performs a shuffle join.

Very slow.

With Broadcast

Spark sends the lookup table once to every Executor.

Each Executor performs the lookup locally.

Benefits

- Faster joins
- No shuffle
- Lower network traffic

---

# When We Used Broadcast in Our Project

Interview Answer

> We had a customer transaction table containing around 500 million records and a small reference table with approximately 500 currency conversion codes. Instead of performing a regular join, we broadcasted the lookup table. This avoided expensive shuffle operations and significantly improved the performance of the ETL pipeline.

---

# 2. Accumulator

## Simple Definition

Accumulator is a variable that Executors can **only add to**.

The Driver reads the final value.

Executors cannot read the Accumulator value.

---

## Real-Life Analogy

Imagine employees putting money into a donation box.

Everyone can add money.

Nobody opens the box.

At the end,

the manager opens it and counts the total.

That's exactly how an Accumulator works.

---

# Architecture

```text
              Driver

Accumulator = 0

      ▲
      │
Executor1 adds 20

Executor2 adds 15

Executor3 adds 30

      │
      ▼

Final Value = 65
```

---

# Example

```python
invalid_records = spark.sparkContext.accumulator(0)
```

Whenever a record is invalid:

```python
invalid_records += 1
```

After processing:

```python
print(invalid_records.value)
```

Output

```
152
```

---

# Real-Time Project Example

Suppose you process

```
10 Million Customer Records
```

Some records have

- Null CustomerID
- Invalid Email
- Missing Phone Number

Every Executor counts invalid rows.

Instead of returning millions of records,

they simply increment the Accumulator.

Finally,

Driver displays

```
Invalid Records

15872
```

---

# Production Example

```python
invalid_count = spark.sparkContext.accumulator(0)

def validate(row):
    global invalid_count

    if row.CustomerID is None:
        invalid_count += 1

df.foreach(validate)

print(invalid_count.value)
```

---

# Interview Answer

> In our project, we used Accumulators to collect data quality metrics such as invalid records, duplicate records, and rejected rows. Each Executor incremented the Accumulator whenever it found an invalid record. After the job completed, the Driver retrieved the final count and stored it in our audit table and PVD dashboard.

---

# Shared Variables

Shared Variables are variables safely shared across the Spark cluster.

They include:

```
Broadcast Variable

Accumulator
```

---

# Comparison

| Feature | Broadcast Variable | Accumulator |
|----------|--------------------|-------------|
| Created By | Driver | Driver |
| Used By | Executors | Executors |
| Read By | Executors | Driver |
| Modified By | No (Read-only) | Yes (Add only) |
| Purpose | Share lookup/reference data | Count or aggregate metrics |
| Common Use | Lookup tables, configuration | Invalid record count, audit metrics |

---

# Real-Time Scenario

Suppose your ETL pipeline processes customer data.

### Step 1

Broadcast

```
Country Lookup

Currency Lookup

Department Lookup
```

Every Executor gets one copy.

---

### Step 2

Executor validates records.

If a record is invalid

```
Accumulator +1
```

---

### Step 3

Driver prints

```
Processed Records

10,000,000

Invalid Records

1,245

Duplicates

22
```

---

# Easy Memory Trick

Think of Spark like a classroom.

**Broadcast Variable**

Teacher gives one book to every student.

Everyone reads the same book.

Nobody changes it.

➡️ Read-only.

---

**Accumulator**

Students put coins into a collection box.

Everyone adds.

Only the teacher opens the box and counts.

➡️ Write from Executors, read by Driver.

---

# 1-Minute Interview Answer

> Spark provides two types of shared variables: Broadcast Variables and Accumulators. A Broadcast Variable is a read-only object created by the Driver and sent once to every Executor. We typically use it for small lookup or reference data to avoid expensive shuffle joins. An Accumulator is created by the Driver and allows Executors to increment values, such as counting invalid or duplicate records. The Driver reads the final value after the job finishes. In one of our ETL projects, we broadcasted small lookup tables like country and currency mappings, and we used Accumulators to track data quality metrics that were displayed in our audit tables and PVD reporting dashboard.